**25 | Final Model Selection and Development Protocol Freeze**

**Human Movement Intelligence | Reza Saadatyar | 2026**

**Aim:** Use only original executed Stages 23–24 to make a transparent, reviewable model-selection recommendation and lock a complete candidate final-training/evaluation protocol. **No new model training, HARTH Test data loading, Test labels, or Test scoring occurs.** This notebook is deliberately a decision-and-provenance document rather than another unplanned benchmark.

**Approval boundary:** The manifest created below remains `PENDING_USER_REVIEW`; it is not a permission to run Test scoring. Stage 26 requires explicit review of the exact frozen plan.

**01. Research Decision, Evidence Sources and Test Boundary**

Stage 23 compared seven classifiers on 14,201 source-identical development windows using three participant-disjoint folds. Stage 24 evaluated two shortlisted architectures under three seeds and isolated three ResNet loss formulations. We must consider global fixed-eight-class Macro F1, per-person variability, Shuffling and both stair classes, compute and transparency. The two original reserved participants S023 and S032 were inspected in early descriptive stages and cannot be described as preregistered-unseen. The goal is a qualified retrospective Test report in Stage 26 only after freezing settings.

In [ ]:
# ==================== 01. Read Verified Saved Development Evidence ====================

# 1. Import research utilities and locate the checked-out repository.
from pathlib import Path
import sys
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
root=Path.cwd().resolve()
while not (root/"code/python/src/core/harth_selection_evidence.py").is_file():
    if root.parent==root: raise RuntimeError("Run within the HMI repository.")
    root=root.parent
sys.path.insert(0,str(root/"code/python/src"))
from core.harth_selection_evidence import load_selection_evidence

# 2. Read previous COMPLETE notebook results; do not copy guessed metrics.
notebooks=root/"studies/wearable_activity_recognition/notebooks"
evidence=load_selection_evidence(
    notebooks/"23_unified_matched_model_benchmark.ipynb",
    notebooks/"24_multiseed_and_imbalance_robustness.ipynb",
)
print("\n"+"="*66)
print("STAGE 25 | DEVELOPMENT-ONLY MODEL SELECTION")
print("="*66)
print("Source SHA256:",evidence["development_sha256"])
print("Stage 23 notebook SHA256:",evidence["notebook23_sha256"])
print("Stage 24 notebook SHA256:",evidence["notebook24_sha256"])
print("Original Test data loaded: NO")
print("="*66)

**02. Seven-Model Development Benchmark and Interpretation**

The complete benchmark is repeated here from saved experimental outputs, not rerun. Different feature representations and loss weighting prevent this ranking from identifying architecture as the sole cause. Stage 24 gives stronger evidence of seed sensitivity for ResNet and TCN, but not equivalent multi-seed replication of Random Forest or the other neural models. All displayed means are from development groups.

In [ ]:
# ==================== 02. Seven-Model Selection Context ====================

# 1. Present all seven original development models without deleting weak runs.
benchmark=evidence["stage23"].sort_values("macro_f1_mean",ascending=False)
display(benchmark.round(4))

# 2. Plot actual saved Macro F1 means and across-fold variability.
fig,ax=plt.subplots(figsize=(11,5))
ax.barh(benchmark["model"],benchmark["macro_f1_mean"],
        xerr=benchmark["macro_f1_sd"],capsize=3)
ax.invert_yaxis()
ax.set(xlim=(0,1),xlabel="Mean held-out-fold fixed-task Macro F1",
       title="Seven-model development comparison (Stage 23)")
ax.grid(axis="x",alpha=.2)
fig.tight_layout()
plt.show()

**03. Model Stability Over Seeds and Fold Assignments**

The Stability analysis compares predeclared seeds `42`, `123` and `2026` under identical 400-step training and three fixed GroupKFold assignments. The average across seed-level three-fold Macro F1 and its standard deviation are descriptive. Reusing the same participants across seeds does not yield nine independent cohorts or validate performance on new activities, sensors or environments.

In [ ]:
# ==================== 03. Seed-Stability and Paired Fold Comparison ====================

# 1. Display all four ResNet/TCN loss conditions, not a post-hoc favourite only.
seed_summary=evidence["stage24_summary"].sort_values(
    "mean_across_seed_fold_macro_f1",ascending=False
)
display(seed_summary.round(4))

# 2. Compare two weighted model families under exactly the same seed/fold grid.
weighted=evidence["stage24_folds"].query("loss_method == 'weighted_ce'")
paired=weighted.pivot(index=["seed","fold"],columns="architecture",values="macro_f1")
paired["ResNet_minus_TCN"]=paired["1D-ResNet"]-paired["TCN"]
display(paired.round(4))
print("Mean matched-fold difference:",
      round(paired["ResNet_minus_TCN"].mean(),4))
print("This comparison is exploratory and development-only.")

**04. Loss Ablation and Minority-Class Trade-offs**

Weighted CE has the best Stage 24 mean Macro F1 for ResNet; Focal and Class-Balanced CE are not automatically superior for rare classes. For original Stage 23 source windows, Random Forest retains higher Shuffling F1 than ResNet, while ResNet achieves higher mean overall Macro F1. The frozen pipeline should therefore include a fixed classical reference, not an after-the-fact attempt to tune on original Test minority outcomes.

In [ ]:
# ==================== 04. Compare Losses and Minority Activity Evidence ====================

# 1. Compare all ResNet training losses across original paired seed/fold runs.
resnet=evidence["stage24_folds"].query("architecture == '1D-ResNet'")
loss_table=resnet.pivot(index=["seed","fold"],columns="loss_method",values="macro_f1")
display(loss_table.round(4))
loss_means=resnet.groupby("loss_method",as_index=False)["macro_f1"].mean()
display(loss_means.sort_values("macro_f1",ascending=False).round(4))

# 2. Preserve original Shuffling and both stair metrics for the key baselines.
class_report=evidence["stage23_classes"].copy()
class_report=class_report[
    class_report["model"].isin(["Random Forest","1D-ResNet","TCN"])
    & class_report["class_code"].isin([3,4,5])
]
display(class_report[[
    "model","class_code","support","precision","recall","f1"
]].round(4))
assert len(class_report)==9

**05. Transparent Model Decision and Alternatives**

**Recommended primary:** 1D-ResNet with inverse-square-root Weighted Cross-Entropy, using seed 42 for the future single fitted model as a reproducible convention, **not** because that seed achieved the highest observed result. **Fixed reference:** Random Forest using exactly the Stage 23 engineering settings. **Second neural comparator:** TCN remains a reported research result but will not be tuned or scored as another candidate in Stage 26 unless the protocol is deliberately reopened *before* seeing Test results.

This choice prioritises the repeated-seed average fixed-eight-class Macro F1. It does not discard the minority Shuffling limitation or imply statistically proven model superiority.

In [ ]:
# ==================== 05. Explicit Development-Informed Model Recommendation ====================

# 1. Verify that the proposed primary method matches the top repeated-seed row.
candidate=seed_summary.iloc[0]
assert candidate["architecture"]=="1D-ResNet"
assert candidate["loss_method"]=="weighted_ce"

# 2. Record the reasoning and non-selected model alternatives for review.
decision=pd.DataFrame([
    {"role":"Primary for proposed Stage 26","architecture":"1D-ResNet",
     "training":"Inverse-square-root Weighted CE","reason":"Highest mean Macro F1 across three seeds and three folds"},
    {"role":"Fixed classical reference","architecture":"Random Forest",
     "training":"Balanced-subsample RF","reason":"Higher observed Shuffling F1 in matched Stage 23"},
    {"role":"Development-only neural comparator","architecture":"TCN",
     "training":"Inverse-square-root Weighted CE","reason":"Strong development results; lower mean across seeds"},
    {"role":"Not selected","architecture":"ResNet Focal / Class-Balanced",
     "training":"Alternative loss ablations","reason":"Lower Stage 24 average fixed-eight-class Macro F1"},
])
display(decision)
print("Primary model candidate: 1D-ResNet + Weighted CE")
print("Proposed final seed: 42 (fixed convention, not best-seed tuning)")
print("No Test-based decision performed.")

**06. Freeze Original Source, Labels, Windowing and Participant Roles**

Lock the official HARTH v2.0 source revision, source six-axis order, 50 Hz timestamps, 100-sample windows, 50-sample stride, strict same-label purity and gap rejection, and eight documented activity codes. Codes outside task remain unmodified in raw files and are excluded **after** time-safe segmentation. The 29 development people remain the only training population for the proposed final model. Original S023/S032 are reserved for one separately qualified test report.

**Important unresolved design detail:** Previous 500-cap studies used chronological, label-blind selection. To keep the future final result directly interpretable in the same bounded task, the *proposed* final fit will use the same 500-cap selection for all 29 development people, rather than silently switching to 132,397 full windows.

In [ ]:
# ==================== 06. Record Immutable Source and Split Conditions ====================

# 1. Explicitly state source provenance, cohort size and feature alignment.
source_protocol=pd.DataFrame([
    {"property":"Source revision","frozen_value":"dad2cfbe89a26f72f19770419469ac037de200df"},
    {"property":"Nominal sampling rate","frozen_value":"50 Hz"},
    {"property":"Window length","frozen_value":"100 samples (2 seconds)"},
    {"property":"Stride","frozen_value":"50 samples"},
    {"property":"Purity / continuity","frozen_value":"100% one documented source label; strict timestamp checks"},
    {"property":"Target class codes","frozen_value":"1, 2, 3, 4, 5, 6, 7, 8"},
    {"property":"Development","frozen_value":"29 people; chronological cap 500; 14,201 windows"},
    {"property":"Original reserved Test","frozen_value":"S023 and S032; no access in Stage 25"},
    {"property":"Original source axes","frozen_value":"Back and thigh accelerometer (three axes each)"},
])
display(source_protocol)
assert len(evidence["development_sha256"])==64

**07. Final Model and Classical Reference Training Settings**

For the proposed single final training run, initialise **one** 1D-ResNet from scratch on all 29 Development subjects, with 400 Adam updates, batch 128, learning rate 0.001 and seed 42. Fit per-channel mean/SD and class weights using *only* those Development windows; all learned statistics must be saved with model weights. Do not select an epoch or tune a threshold using the held-out Test. A separately trained Random Forest uses 30 deterministic statistical features with 120 trees, minimum leaf 3, balanced-subsample weights, seed 42 and no learned Test scaler.

**No probability calibration** is included: Stage 19 did not show consistently positive held-out NLL across folds.

In [ ]:
# ==================== 07. Proposed Final Training Controls ====================

# 1. Make the complete model setup explicit before any future training.
training_config=pd.DataFrame([
    {"setting":"Primary architecture","value":"1D-ResNet, hidden_channels=32"},
    {"setting":"Loss","value":"Fold/Development-only inverse-square-root Weighted CE"},
    {"setting":"Optimiser","value":"Adam, learning_rate=0.001"},
    {"setting":"Budget","value":"400 optimiser updates, batch size=128"},
    {"setting":"Seed","value":"42"},
    {"setting":"CNN preprocessing","value":"Development-only per-axis mean/SD; preserve channel order"},
    {"setting":"RF reference","value":"120 trees, min_samples_leaf=3, balanced_subsample"},
    {"setting":"Final checkpoint","value":"Last update only, no Test-driven selection"},
    {"setting":"Temperature scaling","value":"Not included"},
])
display(training_config)

**08. Final Evaluation Rules, Class Metrics and Restrictions**

The primary reported metric remains fixed-eight-class Macro F1, even if one reserved Test participant lacks certain source classes. Report accuracy, balanced accuracy, confusion matrices, per-class precision/recall/F1 with *true support*, participant-level scores and computational cost. Test prediction must be performed only once following an explicitly approved frozen protocol; Test results cannot be used to choose loss, window size, number of updates, seed, architecture or sampling policy. Because S023/S032 were examined in early descriptive work and come from the same HARTH data collection, this is a **qualified held-out evaluation**, not a fully blind external benchmark.

In [ ]:
# ==================== 08. Evaluation and Non-Leakage Boundaries ====================

# 1. Document what may and may not change after protocol approval.
rules=pd.DataFrame([
    {"item":"Primary evaluation","fixed_rule":"Eight-class Macro F1 with original source class order"},
    {"item":"Secondary outcomes","fixed_rule":"Balanced accuracy, accuracy, class precision/recall/F1 and support"},
    {"item":"Participant reporting","fixed_rule":"Scores for both original reserved participants separately"},
    {"item":"Model comparisons","fixed_rule":"One primary ResNet and one fixed RF reference; no Test-guided choice"},
    {"item":"Preprocessing","fixed_rule":"Fit only on 29-person Development"},
    {"item":"Selection and stopping","fixed_rule":"400 final updates; no Test-based checkpoint selection"},
    {"item":"Unmapped labels","fixed_rule":"Preserve original codes; exclude post-windowing"},
    {"item":"Evidence limitation","fixed_rule":"Previously inspected reserved people, not strictly blind external Test"},
])
display(rules)

**09. Code Fingerprints and Approval-Pending Protocol Manifest**

A freeze is credible only if another researcher can identify exactly which code, numerical settings, label definitions and prior development evidence support it. This section generates a JSON **proposal** with SHA256 hashes of the source modules and executed notebooks. The manifest stays `PENDING_USER_REVIEW` until explicitly approved; it is not automatically treated as final. This stage does not write model weights or inspect reserved recordings.

In [ ]:
# ==================== 09. Generate Versioned Protocol Proposal ====================

# 1. Hash the exact source modules that implement this pipeline.
paths=[
    "code/python/src/core/advanced_temporal_models.py",
    "code/python/src/core/temporal_cnn.py",
    "code/python/src/core/temporal_benchmark.py",
    "code/python/src/core/feature_engineering.py",
    "code/python/src/core/robustness_experiments.py",
    "studies/wearable_activity_recognition/src/harth_data.py",
]
code_sha={}
for name in paths:
    file=root/name
    assert file.is_file(),name
    code_sha[name]=hashlib.sha256(file.read_bytes()).hexdigest()

# 2. Compose a complete selection record without claiming Test measurements.
manifest={
    "schema":"harth-final-protocol-v1",
    "status":"PENDING_USER_REVIEW",
    "source_development_sha256":evidence["development_sha256"],
    "stage23_saved_notebook_sha256":evidence["notebook23_sha256"],
    "stage24_saved_notebook_sha256":evidence["notebook24_sha256"],
    "primary_model":{
        "architecture":"1D-ResNet","hidden_channels":32,"in_channels":6,
        "classes":list(range(1,9)),"training_loss":"inverse_sqrt_weighted_ce",
        "random_seed":42,"optimizer":"Adam","learning_rate":0.001,
        "batch_size":128,"optimizer_steps":400,"checkpoint":"final_step",
        "preprocessing":"Development-only per-channel mean and SD",
    },
    "reference_model":{
        "architecture":"Random Forest","n_estimators":120,
        "min_samples_leaf":3,"class_weight":"balanced_subsample",
        "seed":42,"features":["mean","std","rms","peak_to_peak","waveform_length"],
    },
    "source":{
        "dataset":"HARTH v2.0",
        "revision":"dad2cfbe89a26f72f19770419469ac037de200df",
        "sampling_rate_hz":50,"window_size":100,"step_size":50,
        "annotation_purity":1.0,"timestamp_continuity":True,
        "task_codes":list(range(1,9)),
        "development_participants":29,"cap_per_development_person":500,
        "expected_development_windows":14201,
        "reserved_test_people":["S023","S032"],
        "test_cap_per_person":500,
        "unknown_source_codes":"Preserve and exclude after time-safe windowing",
    },
    "evaluation":{
        "main_metric":"Fixed-eight-class Macro F1",
        "additional":["accuracy","balanced_accuracy",
                      "per-class precision/recall/F1/support",
                      "per-person fixed-task Macro F1","confusion matrices"],
        "use_test_for_tuning":False,
        "temperature_scaling":False,
        "description":"Qualified previously-inspected reserved Test, not fully blind external validation",
    },
    "module_sha256":code_sha,
}

# 3. Save a reviewable proposed JSON only if no conflicting copy exists.
target=root/"studies/wearable_activity_recognition/experiments/stage_25_protocol_proposal.json"
target.parent.mkdir(parents=True,exist_ok=True)
payload=json.dumps(manifest,indent=2,ensure_ascii=False)+"\n"
if target.exists() and target.read_text(encoding="utf-8")!=payload:
    raise FileExistsError("Existing reviewed protocol differs: do not overwrite.")
target.write_text(payload,encoding="utf-8")
print("Proposed protocol path:",target.relative_to(root))
print("Approval status:",manifest["status"])
print("Code module SHA256 entries:",len(code_sha))
print("Test recordings read during Stage 25: NO")

**10. Research Recommendation, Alternatives and Explicit User Review**

This section makes the development-informed decision understandable without converting it into an unsupported superiority claim. The primary ResNet showed higher average fixed-eight-class Macro F1 across three seeds and held-out folds, while RF remains particularly informative for Shuffling. A strong minority-class trade-off remains, and further model training or test-based adjustment must not occur as part of a silent selection procedure.

**Required review before Stage 26:** Confirm primary model, reference RF, 500-window per-person cap, use of both previously descriptively inspected reserved participants for a *qualified* held-out comparison, and full metric reporting. If changes are desired, revise the proposal now, before any Test prediction.

In [ ]:
# ==================== 10. Model-Selection Decision and Open Review Items ====================

# 1. Present the frozen candidate and alternatives with reviewable rationale.
review=pd.DataFrame([
    {"item":"Primary final candidate","proposal":"1D-ResNet + Weighted CE","requires_approval":True},
    {"item":"Classic reference","proposal":"Random Forest 120-tree","requires_approval":True},
    {"item":"Bounded cohort","proposal":"500 source windows/person","requires_approval":True},
    {"item":"Held-out set","proposal":"S023, S032 (historically inspected)","requires_approval":True},
    {"item":"Single final seed","proposal":"42, fixed convention","requires_approval":True},
    {"item":"Allowed scoring","proposal":"One qualified held-out evaluation after approval","requires_approval":True},
])
display(review)
print("Development evidence supports this proposed configuration.")
print("Explicit user approval before Stage 26: REQUIRED")

**11. Final Scientific Quality Gate: Proposed Freeze, No Test Use**

A successful quality gate demonstrates that all seven original model-family comparisons and 36 follow-up robustness runs are evidenced by completed, matching original Development notebooks; that the proposed settings are explicit; and that no Test data were accessed in this decision stage. It **does not** approve a Test run or create model weights. The final manifest must be reviewed before Stage 26 begins.

In [ ]:
# ==================== 11. Scientific Protocol Freeze Quality Gate ====================

# 1. Verify complete previous evidence, provenance and candidate identity.
assert len(evidence["stage23"])==7
assert len(evidence["stage24_folds"])==36
assert len(evidence["stage24_summary"])==4
assert manifest["status"]=="PENDING_USER_REVIEW"
assert manifest["source_development_sha256"]==evidence["development_sha256"]
assert len(manifest["module_sha256"])==6

# 2. Verify the primary settings and protected subject boundaries.
assert manifest["primary_model"]["architecture"]=="1D-ResNet"
assert manifest["primary_model"]["random_seed"]==42
assert manifest["primary_model"]["optimizer_steps"]==400
assert manifest["source"]["reserved_test_people"]==["S023","S032"]
assert not manifest["evaluation"]["use_test_for_tuning"]
assert manifest["evaluation"]["main_metric"]=="Fixed-eight-class Macro F1"

# 3. Provide a truthful, reader-friendly completion summary.
print("\n"+"="*70)
print("STAGE 25: FINAL MODEL SELECTION AND PROTOCOL PROPOSAL")
print("="*70)
print("Source-matched completed Development evidence: PASSED")
print("Seven-model Stage 23 evaluation reviewed: PASSED")
print("Three-seed/loss Stage 24 evidence reviewed: PASSED")
print("Primary development candidate: 1D-ResNet + Weighted CE")
print("Fixed reference model: Random Forest")
print("Protocol module/source digests: RECORDED")
print("Protected Test subjects loaded: NO")
print("-"*70)
print("Stage 25 development selection audit: PASSED")
print("Protocol status: PENDING USER REVIEW")
print("Test evaluation performed: NO")
print("Next: user protocol approval before Stage 26")
print("="*70)